In [1]:
import os, sys

if not os.path.isdir("Evaluation"):
    os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("cwd:", os.getcwd())
print("Evaluation/ exists:", os.path.isdir("Evaluation"))


cwd: c:\Users\Offic\OneDrive\Desktop\Projects\5 - XAI Evaluation Paper
Evaluation/ exists: True


## Experimental Setup

In [2]:
from src.evaluation import create_experiment
from src.stability import run_stability
from src.utils import Logger

os.makedirs("Evaluation/logs", exist_ok=True)

DATASETS = [
    ("healthcare", "pima_diabetes"),
    ("healthcare", "breast_cancer_wisconsin"),
    ("healthcare", "heart_disease_uci"),
    ("finance", "loan_default"),
    ("finance", "financial_distress"),
    ("finance", "credit_card_fraud_2023"),
]
MODEL_NAMES = ["rf", "xgb"]

logger = Logger("Evaluation/logs", filename="ledger.log")
experiment_id = create_experiment(
    "Evaluation",
    {"phase": "stability", "datasets": DATASETS, "models": MODEL_NAMES},
    logger,
)
print("experiment_id:", experiment_id)

Created experiment_id=exp_20260719T201110_b5ef64ad, logged to Evaluation\run_ledger.csv
experiment_id: exp_20260719T201110_b5ef64ad


In [3]:
results = {}
for domain, dataset_name in DATASETS:
    for model_name in MODEL_NAMES:
        run_dir = os.path.join("Explanations", domain, dataset_name, model_name)
        if not os.path.isdir(run_dir):
            print(f"SKIP {domain}/{dataset_name}/{model_name}: no Explanations/ artifacts yet")
            continue
        try:
            rows = run_stability({
                "domain": domain,
                "dataset_name": dataset_name,
                "model_name": model_name,
                "experiment_id": experiment_id,
                "stability_root": "Evaluation/Stability",
            })
            results[(domain, dataset_name, model_name)] = len(rows)
        except Exception as e:
            print(f"FAILED {domain}/{dataset_name}/{model_name}: {e}")

print()
print(f"{len(results)}/{len(DATASETS) * len(MODEL_NAMES)} combos completed.")
results


------------------------------------------------------------
STABILITY: pima_diabetes x rf
------------------------------------------------------------
Loaded model (skops): Models\healthcare\pima_diabetes\rf\model\model.skops
Feature-group map for pima_diabetes: 8 original features (from 8 model-input columns).
Wrote 1232 rows to Evaluation/Stability\healthcare\pima_diabetes\rf\shap\repeated_attributions.csv
Wrote 12320 rows to Evaluation/Stability\healthcare\pima_diabetes\rf\lime\repeated_attributions.csv
Appended 616 row(s) to Evaluation\metrics_long.csv

------------------------------------------------------------
DONE
------------------------------------------------------------
Stability: 616 metrics_long rows written across ['shap', 'lime'].

------------------------------------------------------------
STABILITY: pima_diabetes x xgb
------------------------------------------------------------
Loaded model (skops): Models\healthcare\pima_diabetes\xgb\model\model.skops (trusted ty

{('healthcare', 'pima_diabetes', 'rf'): 616,
 ('healthcare', 'pima_diabetes', 'xgb'): 616,
 ('healthcare', 'breast_cancer_wisconsin', 'rf'): 456,
 ('healthcare', 'breast_cancer_wisconsin', 'xgb'): 456,
 ('healthcare', 'heart_disease_uci', 'rf'): 736,
 ('healthcare', 'heart_disease_uci', 'xgb'): 736,
 ('finance', 'loan_default', 'rf'): 2000,
 ('finance', 'loan_default', 'xgb'): 2000,
 ('finance', 'financial_distress', 'rf'): 2936,
 ('finance', 'financial_distress', 'xgb'): 2936,
 ('finance', 'credit_card_fraud_2023', 'rf'): 2000,
 ('finance', 'credit_card_fraud_2023', 'xgb'): 2000}

## Preview results

In [4]:
import pandas as pd
import os

def discover_stability_combos(stability_root="Evaluation/Stability"):
    combos = []
    if not os.path.isdir(stability_root):
        return combos
    for domain in sorted(os.listdir(stability_root)):
        d_path = os.path.join(stability_root, domain)
        if not os.path.isdir(d_path): continue
        for dataset in sorted(os.listdir(d_path)):
            ds_path = os.path.join(d_path, dataset)
            if not os.path.isdir(ds_path): continue
            for model in sorted(os.listdir(ds_path)):
                m_path = os.path.join(ds_path, model)
                for explainer in sorted(os.listdir(m_path)):
                    f = os.path.join(m_path, explainer, "repeated_attributions.csv")
                    if os.path.exists(f):
                        combos.append((domain, dataset, model, explainer, f))
    return combos

combos = discover_stability_combos()
print(f"Found {len(combos)} combo(s) with raw stability artifacts.\n")

agg_rows = []
for domain, dataset, model, explainer, path in combos:
    raw = pd.read_csv(path)
    n_repeats_found = raw["repeat_idx"].nunique()
    agg_rows.append({
        "domain": domain, "dataset": dataset, "model": model, "explainer": explainer,
        "n_repeats": n_repeats_found,
        "mean_attribution_std_across_repeats": (
            raw.groupby("feature")["attribution"].std().mean() if n_repeats_found > 1 else 0.0
        ),
    })

agg_df = pd.DataFrame(agg_rows).sort_values(["domain", "dataset", "model", "explainer"])
agg_df

Found 24 combo(s) with raw stability artifacts.



,domain,dataset,model,explainer,n_repeats,mean_attribution_std_across_repeats
0,finance,credit_card_fraud_2023,rf,lime,10,0.012047
1,finance,credit_card_fraud_2023,rf,shap,1,0.000000
2,finance,credit_card_fraud_2023,xgb,lime,10,0.028787
3,finance,credit_card_fraud_2023,xgb,shap,1,0.000000
4,finance,financial_distress,rf,lime,10,0.006249
5,finance,financial_distress,rf,shap,1,0.000000
6,finance,financial_distress,xgb,lime,10,0.015227
7,finance,financial_distress,xgb,shap,1,0.000000
8,finance,loan_default,rf,lime,10,0.008809
9,finance,loan_default,rf,shap,1,0.000000


## Pull Stability lng from metrics_long.csv

In [5]:
metrics_long = pd.read_csv("Evaluation/metrics_long.csv")

stability_scores = metrics_long[
    (metrics_long["metric_property"] == "Stability")
    & (metrics_long["experiment_id"] == experiment_id)
]

print(f"{len(stability_scores)} Stability rows for this experiment_id.")

# Flag anything that didn't resolve cleanly (degenerate LIME instances)
non_ok = stability_scores[stability_scores["status"] != "ok"]
if len(non_ok):
    print(f"\n{len(non_ok)} row(s) with status != 'ok':")
    print(non_ok.groupby(["domain", "dataset", "model", "explainer", "status"]).size())

# Per-combo mean (excluding non-ok rows so degenerate instances don't
# silently pull the mean toward None/NaN)
summary = (
    stability_scores[stability_scores["status"] == "ok"]
    .groupby(["domain", "dataset", "model", "explainer", "metric_name"])["value"]
    .mean()
    .unstack("metric_name")
    .sort_index()
)
summary

17488 Stability rows for this experiment_id.


C:\Users\Offic\AppData\Local\Temp\ipykernel_21968\175427483.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  metrics_long = pd.read_csv("Evaluation/metrics_long.csv")


metric_name                                         mean_pairwise_spearman_rank_correlation  \
domain     dataset                 model explainer                                            
finance    credit_card_fraud_2023  rf    lime                                      0.583806   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.515479   
                                         shap                                      1.000000   
           financial_distress      rf    lime                                      0.500487   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.375942   
                                         shap                                      1.000000   
           loan_default            rf    lime                                      0.926502   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.624712   
                                         shap                                      1.000000   
healthcare breast_cancer_wisconsin rf    lime                                      0.742225   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.758570   
                                         shap                                      1.000000   
           heart_disease_uci       rf    lime                                      0.955649   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.924359   
                                         shap                                      1.000000   
           pima_diabetes           rf    lime                                      0.852769   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.830571   
                                         shap                                      1.000000   

metric_name                                         mean_pairwise_top_4_jaccard_overlap  \
domain     dataset                 model explainer                                        
finance    credit_card_fraud_2023  rf    lime                                       NaN   
                                         shap                                       NaN   
                                   xgb   lime                                       NaN   
                                         shap                                       NaN   
           financial_distress      rf    lime                                       NaN   
                                         shap                                       NaN   
                                   xgb   lime                                       NaN   
                                         shap                                       NaN   
           loan_default            rf    lime                                       NaN   
                                         shap                                       NaN   
                                   xgb   lime                                       NaN   
                                         shap                                       NaN   
healthcare breast_cancer_wisconsin rf    lime                                       NaN   
                                         shap                                       NaN   
                                   xgb   lime                            

In [6]:
import re

metrics_long = pd.read_csv("Evaluation/metrics_long.csv")

stability_scores = metrics_long[
    (metrics_long["metric_property"] == "Stability")
    & (metrics_long["experiment_id"] == experiment_id)
].copy()

print(f"{len(stability_scores)} Stability rows for this experiment_id.")

non_ok = stability_scores[stability_scores["status"] != "ok"]
if len(non_ok):
    print(f"\n{len(non_ok)} row(s) with status != 'ok':")
    print(non_ok.groupby(["domain", "dataset", "model", "explainer", "status"]).size())

stability_scores = stability_scores[stability_scores["status"] == "ok"].copy()

# Pull k out of "mean_pairwise_top_5_jaccard_overlap" -> k_used=5,
# metric_family="jaccard_overlap"; leave the correlation metric_name as-is.
# This is what makes datasets with different n_features (and therefore
# different k) land in the SAME column instead of NaN-fragmenting across
# top_4/top_5/etc. — see STABILITY_README.md §8 on the k-normalization
# deferral.
def parse_metric_family(name):
    m = re.match(r"mean_pairwise_top_(\d+)_jaccard_overlap", name)
    if m:
        return "mean_pairwise_top_k_jaccard_overlap", int(m.group(1))
    return name, None

parsed = stability_scores["metric_name"].apply(parse_metric_family)
stability_scores["metric_family"] = parsed.apply(lambda x: x[0])
stability_scores["k_used"] = parsed.apply(lambda x: x[1])

summary = (
    stability_scores
    .groupby(["domain", "dataset", "model", "explainer", "metric_family"])["value"]
    .mean()
    .unstack("metric_family")
    .sort_index()
)

# k is constant within a (dataset) since it's driven by n_features, not by
# model/explainer/instance -- surface it as its own column for visibility
# rather than losing it once metric_family is unified.
k_by_dataset = (
    stability_scores[stability_scores["k_used"].notna()]
    .groupby(["domain", "dataset"])["k_used"]
    .agg(lambda s: s.mode().iloc[0])
    .rename("k_used")
)

summary = summary.join(k_by_dataset, on=["domain", "dataset"])
summary

17488 Stability rows for this experiment_id.


C:\Users\Offic\AppData\Local\Temp\ipykernel_21968\393605736.py:3: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  metrics_long = pd.read_csv("Evaluation/metrics_long.csv")


mean_pairwise_spearman_rank_correlation  \
domain     dataset                 model explainer                                            
finance    credit_card_fraud_2023  rf    lime                                      0.583806   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.515479   
                                         shap                                      1.000000   
           financial_distress      rf    lime                                      0.500487   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.375942   
                                         shap                                      1.000000   
           loan_default            rf    lime                                      0.926502   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.624712   
                                         shap                                      1.000000   
healthcare breast_cancer_wisconsin rf    lime                                      0.742225   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.758570   
                                         shap                                      1.000000   
           heart_disease_uci       rf    lime                                      0.955649   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.924359   
                                         shap                                      1.000000   
           pima_diabetes           rf    lime                                      0.852769   
                                         shap                                      1.000000   
                                   xgb   lime                                      0.830571   
                                         shap                                      1.000000   

                                                    mean_pairwise_top_k_jaccard_overlap  \
domain     dataset                 model explainer                                        
finance    credit_card_fraud_2023  rf    lime                                  0.818543   
                                         shap                                  1.000000   
                                   xgb   lime                                  0.633697   
                                         shap                                  1.000000   
           financial_distress      rf    lime                                  0.560696   
                                         shap                                  1.000000   
                                   xgb   lime                                  0.530323   
                                         shap                                  1.000000   
           loan_default            rf    lime                                  0.824313   
                                         shap                                  1.000000   
                                   xgb   lime                                  0.692816   
                                         shap                                  1.000000   
healthcare breast_cancer_wisconsin rf    lime                                  0.763675   
                                         shap                                  1.000000   
                                   xgb   lime                                  0.603799   
                                  